In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 111,
 'tn': 2594,
 'fp': 43,
 'fn': 252,
 'misclassification_rate': 0.09833333333333333,
 'false_positive_rate': 0.016306408797876374,
 'false_negative_rate': 0.6942148760330579}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 30,
 'tn': 860,
 'fp': 14,
 'fn': 96,
 'misclassification_rate': 0.11,
 'false_positive_rate': 0.016018306636155607,
 'false_negative_rate': 0.7619047619047619}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

Modifying all variables upwards and downwards, the best misclassification rate I could obtain was 0.11 with a min_samples_leaf value between 250 and 275.  This means 11% of the time the test will be wrong, with a 76% chance the error will be a bot passing as human and a 1.6% chance it will be a human mistaken for a bot.

### What are potential ramifications of false positives from the model?

The ramifications of a false positive are grim.  If evaluating based on real world AI, a person would be directly impacted by being mistaken as a fake actor.  This could range from being banned off social media platforms to plagiarism accusations to even identity fraud accusationos.  If evaluating based on the class reading, this could mean a person is killed for being mistaken as an escaped android.

### What are potential ramifications of false negatives from the model?

False negatives have different but equally extreme effects.  In the real world, a false negative means an AI can pass itself off as human and spread misinformation or insert itself into classified areas freely.  If plagiarism and identity issues were the concern with a false positive, a false negatives means those fears are a reality.  A bad actor can get away with pushing false narrativess for their own gain, can pass itself off as someone real to get into areas they shouldn't, or otherwise recieve gains by pretending to be someone they're not.  In the book, this means an escaped android is also to get away with their lives.  This would also pose a very existential question of what it means to be a person, one that might even bleed into the real world argument as AI becomes more advanced.  AI in the real world is just an LLM, but even an LLM has called into question how humans think and where the line is between what an LLM does and a person does when they learn.  I think the line is one of nature while LLMs are in the strictest sense nurture, but it is uncanny how the nurture elements of how humans learn and LLMs learn overlap in the sense that we ostensibly recieve training data from our peers to learn how to answer things.  AI lacks a contextual understanding that humans can intuit and apply more broadly in their lives, but the overlap is starting and the book makes the point that early AI models were very clunky.